# Optional: 真实模型如何穿过 AgentKernel Tool Boundary？

## Opt-in Real Model Tool Trace

这个 notebook 默认不会调用网络或 provider。只有显式设置 `AGENTKERNEL_RUN_REAL_MODEL=1` 以及 OpenAI-compatible provider 环境变量时，才会运行真实模型。

It exposes observable runtime facts only: model-visible request summary, tool proposal, Kernel authorization, ToolResult, Session events, and final answer. It does not request or display private chain-of-thought.

In [ ]:
from pathlib import Path
import sys

def find_repo_root(start=Path.cwd()):
    for path in (start, *start.parents):
        if (path / "agentkernel").is_dir():
            return path
    raise RuntimeError("Run this notebook from inside the AgentKernel repository.")

REPOSITORY_ROOT = find_repo_root()
if str(REPOSITORY_ROOT) not in sys.path:
    sys.path.insert(0, str(REPOSITORY_ROOT))
LABS_ROOT = REPOSITORY_ROOT / "examples" / "labs"
if str(LABS_ROOT) not in sys.path:
    sys.path.insert(0, str(LABS_ROOT))

from lab_helpers import event_rows, grant_rows, print_table, process_row, trajectory

## 1. Provider configuration gate

In [ ]:
import os

RUN_REAL = os.environ.get("AGENTKERNEL_RUN_REAL_MODEL") == "1"
print_table([
    {"variable": "AGENTKERNEL_RUN_REAL_MODEL", "value": os.environ.get("AGENTKERNEL_RUN_REAL_MODEL", "<unset>")},
    {"variable": "AGENTKERNEL_LLM_BASE_URL", "value": "<set>" if os.environ.get("AGENTKERNEL_LLM_BASE_URL") else "<unset>"},
    {"variable": "AGENTKERNEL_LLM_MODEL", "value": os.environ.get("AGENTKERNEL_LLM_MODEL", "<unset>")},
    {"variable": "AGENTKERNEL_LLM_API_KEY", "value": "<set>" if os.environ.get("AGENTKERNEL_LLM_API_KEY") else "<unset optional>"},
])
if not RUN_REAL:
    print("SKIPPED: no network/provider call will be made.")

## 2. Run only when explicitly enabled

In [ ]:
if RUN_REAL:
    import asyncio
    from examples.real_agent.basic_tool_trace import run

    exit_code = asyncio.run(run(trace_jsonl=None))
    print_table([{"real_model_demo_exit_code": exit_code}])
else:
    print("Real-model execution skipped. Set AGENTKERNEL_RUN_REAL_MODEL=1 to opt in.")

## Observable trajectory

Task → model-visible request → model response/tool proposal → Kernel authorization → ToolResult → next model request → final answer → Session facts.

## WHAT THIS DEMONSTRATES / 本实验验证什么

- With explicit provider configuration, a real model can receive Tool schemas and propose Tool calls through AgentKernel.
- The Kernel boundary remains observable without exposing private chain-of-thought.

## WHAT THIS DOES NOT DEMONSTRATE / 本实验不证明什么

- It does not run in CI by default.
- It does not prove model reasoning quality, production reliability, production security, or exactly-once side effects.
- It does not make any network call unless explicitly opted in.